# Library & Data

In [ ]:
# If running in Colab, mount Google Drive here.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import gc
import matplotlib.pyplot as plt

from tqdm import tqdm

import tensorflow as tf
from sklearn.model_selection import train_test_split

from lightgbm import LGBMClassifier

In [ ]:
train=pd.read_csv('../data/train.csv')
vali = pd.read_csv('../data/validation_sample.csv')
test=pd.read_csv('../data/test.csv')
submission=pd.read_csv('../data/sample_submission.csv')

# 데이터 EDA 

## 초기) 간단 EDA

In [ ]:
print("Train의 형태: ",train.shape)

In [ ]:
train.info()

In [ ]:
train.head()

In [ ]:
pd.DataFrame(train['full_log'].value_counts())

In [ ]:
#train full_log의 길이 확인
train['full_log'].str.split(' ').str.len().hist(bins=50)

In [ ]:
#train level별 값 확인
train['level'].value_counts()

## 기본) 레벨에 따른 단어 분리

In [ ]:
train

In [ ]:
class letssee:
  def __init__(self, data):
    train_df = data

    first_strings = pd.merge(train_df.loc[train_df.level!=0].full_log.apply(lambda x: x.split(' ')[0]).value_counts().rename('lv!=0'),
                         train_df.loc[train_df.level==0].full_log.apply(lambda x: x.split(' ')[0]).value_counts().rename('lv==0'),
                         how='outer',left_index=True,right_index=True).fillna(0).astype(int)

    self.first_strings = first_strings.sort_values('lv==0',ascending=False)

  def graph(self): 
    first_strings = self.first_strings
    fig,ax = plt.subplots(1,1,figsize=(12,7))
    ax.bar([i-0.25 for i in range(first_strings.shape[0])], first_strings.values[:,1],width=0.4)
    ax.bar([i+0.25 for i in range(first_strings.shape[0])], first_strings.values[:,0],width=0.4)
    ax.set_yscale("log")
    ax.set_xticks(range(first_strings.shape[0]))
    ax.set_xticklabels(first_strings.index, rotation=70)
    ax.legend(['Level == 0','Level != 0'])
    fig.show()

  def show(self):
    first_strings = self.first_strings
    print(first_strings)


In [ ]:
a = letssee(train)

In [ ]:
a.graph()

In [ ]:
a.show()

## 심화1) 숫자가 의미가 있을까?
* https://dacon.io/competitions/official/235717/codeshare/2637?page=1&dtype=recent 참조

In [ ]:
# 기존
pd.DataFrame(train['full_log'].value_counts())

In [ ]:
train1 = train.copy()
# text 전처리로 문자만 남김

train1['full_log'] = train1['full_log'].str.replace(pat=r'[^A-Za-z]', repl=r'', regex=True)

In [ ]:
# 문자 제거
train1['full_log'].value_counts()

동일한 Text가 다수 존재 (클러스터링)

## 심화1-1) 동일한 Text는 어떤 의미를 가지는 가?

In [ ]:
train_df = train1
b = letssee(train_df)

In [ ]:
b.show()

* 동일한 Text는 동일한 Level을 가짐
* 숫자는 크게 중요하지 않음

## 심화2) 숫자와 월을 변경해보자

In [ ]:
train2 = train.copy()

#full_log에서 숫자는 마스킹 처리
train2['full_log']=train2['full_log'].str.replace(r'[0-9,Jan,Feb,Dec,Sep,Oct,Nov,Mar]', 'na')

In [ ]:
# 숫자와 해당 월을 제거 후
train2['full_log'].value_counts()

In [ ]:
c = letssee(train2)

In [ ]:
c.graph()

In [ ]:
c.show()

## 심화2-1) 숫자와 월을 제거해보자

In [ ]:
train2 = train.copy()

#full_log에서 숫자는 마스킹 처리
train2['full_log']=train2['full_log'].str.replace(r'[0-9,Jan,Feb,Dec,Sep,Oct,Nov,Mar]', '')

In [ ]:
# 숫자와 해당 월을 제거 후
train2['full_log'].value_counts()

In [ ]:
c = letssee(train2)

In [ ]:
c.graph()

In [ ]:
c.show()

## 심화2-2) 여러가지 조합

In [ ]:
train2 = train.copy()

#full_log에서 숫자는 마스킹 처리
train2['full_log']=train2['full_log'].str.replace(r'[0-9,Jan,Feb,Dec,Sep,Oct,Nov,Mar]', '')

In [ ]:
# 문자열 제거
train2['full_log'] = train2['full_log'].str.replace(pat=r'[^A-Za-z]', repl=r' ', regex=True)

In [ ]:
# 숫자와 해당 월을 제거 후
train2['full_log'].value_counts()

In [ ]:
c = letssee(train2)

In [ ]:
c.graph()

In [ ]:
c.show()

> 대회 규칙이 "패턴 매칭 알고리즘 사용 불가"
* 패턴 매칭 알고리즘 사용 불가는 규칙기반 모델이 아닌 학습기반의 모델만 사용가능하다는 의미
* tag를 추출하여 학습 feature로 사용하는 것은 가능하나 추출한 tag와 사람이 지정한 조건문만을 이용한 예측은 불가

# 데이터 전처리

In [ ]:
# 홀드아웃방식으로 Train과 Test로 분리
text_train, text_val, label_train, label_val = train_test_split(train['full_log'], train['level'], test_size=0.2, random_state=42)

In [ ]:
# 각 데이터에 대하여 토큰화 처리
tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(text_train)
top_k = len(tokenizer.word_index)

x_train = tokenizer.texts_to_sequences(text_train)
x_val = tokenizer.texts_to_sequences(text_val)

max_length=100

x_train_vector = tf.keras.preprocessing.sequence.pad_sequences(
    x_train, maxlen=max_length, padding='post'
)

x_val_vector = tf.keras.preprocessing.sequence.pad_sequences(
    x_val, maxlen=max_length, padding='post'
)

In [ ]:
# 토큰화된 훈련 데이터 확인
x_train_vector.shape

In [ ]:
# 토큰화된 테스트 데이터 확인
x_val_vector.shape

In [ ]:
# 토큰화된 훈련 데이터 확인2
x_train_vector

# 모델링

LGBM 모델

In [ ]:
# LGBM 모델
lgb = LGBMClassifier(n_estimators=400, random_state=42)
evals = [(x_val_vector, label_val)]
lgb.fit(x_train_vector, label_train, early_stopping_rounds=50, eval_metric='error', eval_set=evals, verbose=True)

XGBoost 모델

In [ ]:
# XGBoost 모델
import xgboost as xgb
xgb = xgb.XGBClassifier(n_estimators=1500, random_state=42, n_jobs=-1)
evals = [(x_val_vector, label_val)]
xgb.fit(x_train_vector, label_train, early_stopping_rounds=50, eval_metric='merror', eval_set=evals, verbose=True)

* XGBClassifier
              (base_score=0.5, booster='gbtree', colsample_bylevel=1,
              colsample_bynode=1, colsample_bytree=1, gamma=0,
              learning_rate=0.1, max_delta_step=0, max_depth=3,
              min_child_weight=1, missing=None, n_estimators=1500, n_jobs=1,
              nthread=None, objective='multi:softprob', random_state=42,
              reg_alpha=0, reg_lambda=1, scale_pos_weight=1, seed=None,
              silent=None, subsample=1, verbosity=1)

* 0.9472021192302403

# 모델 성능 확인

In [ ]:
# XGBoost 모델 자체
xgb.score(x_val_vector, label_val)

In [ ]:
# crosstab으로 확인
pred=xgb.predict(x_val_vector)
crosstab = pd.crosstab(label_val, pred, rownames=['real'], colnames=['pred'])
crosstab

In [ ]:
# 검증 산식
def macro_f1(answer_df, submission_df):    
    true = answer_df
    pred = submission_df

    score = metrics.f1_score(y_true=true, y_pred=pred, average='macro')
    return score    

In [ ]:
# 모델 예측
preds=xgb.predict(x_val_vector)
probas=xgb.predict_proba(x_val_vector)
print(preds.shape)
print(probas.shape)

In [ ]:
# 검증 산식을 통한 검증 지표
macro_f1(label_val,preds)

In [ ]:
# 오차행렬 제시
get_clf_eval(label_val,preds)

# 제출

In [ ]:
#test 데이터 전처리
text_test = test['full_log'].str.replace(r'[0-9,Jan,Feb,Dec,Sep,Oct,Nov,Mar]', '')

x_test = tokenizer.texts_to_sequences(text_test)

x_test_vector = tf.keras.preprocessing.sequence.pad_sequences(
   x_test , maxlen=max_length, padding='post'
)

In [ ]:
# 학습된 XGBoost 모델을 통해 제출물 산출
results=xgb.predict(x_test_vector)
results_proba=xgb.predict_proba(x_test_vector)
# results[np.where(np.max(results_proba, axis=1) < 0.9)] = 7

+ 새로운 위험요소에 대한 가정 추가
+ => 예측치의 예측 확률이 0.90이하인 경우, 즉 확신이 없을 경우 이상치로 판단 

In [ ]:
# 제출물 형식에 맞추어 생성
submission['level']=results

In [ ]:
# 최종 제출물 확인
submission

In [ ]:
# 제출물 제출
submission.to_csv('../outputs/submission_lgb2_0512.csv', index=False)